# Groundtruth — Days 1–2 in Google Colab

**Before you start**
1. Upload `groundtruth.zip` to the top level of your Google Drive (**My Drive**).
2. In Colab: **Runtime → Change runtime type → T4 GPU** (free). Embedding is ~20× faster than on CPU.
3. Run the cells top to bottom. Everything you create (corpus, golden set, results) is saved to `My Drive/groundtruth`, so it survives when Colab disconnects. When you come back, rerun cells 1–3 only.

## 1. Connect Google Drive and unzip the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
PROJECT = '/content/drive/MyDrive/groundtruth'
ZIP = '/content/drive/MyDrive/groundtruth.zip'
if not os.path.isdir(PROJECT):
    assert os.path.isfile(ZIP), 'Upload groundtruth.zip to the top of My Drive first'
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/drive/MyDrive')
    print('Unzipped to', PROJECT)
else:
    print('Project already in Drive, keeping your existing work:', PROJECT)
%cd {PROJECT}

## 2. Install dependencies
Colab already has PyTorch, so this only adds what's missing. Rerun this cell after every new Colab session.

In [ ]:
!pip install -q -r requirements.txt
import torch, sentence_transformers, sys
sys.path.insert(0, 'src')  # make the groundtruth package importable in this notebook
print('python', sys.version.split()[0], '| sentence-transformers', sentence_transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU')

## 3. Run the tests
Expected: `56 passed, 1 skipped` (the skipped one is the CI gate, which switches on once real data exists).

In [ ]:
!python -m pytest -q -rs

## 4. Download one year of Supreme Court judgments
Source: [Indian Supreme Court Judgments](https://registry.opendata.aws/indian-supreme-court-judgments/), CC-BY-4.0. Downloaded to Colab's fast local disk (not Drive); only the cleaned text goes to Drive. The first line prints the file size so you can see what you're downloading.

In [ ]:
YEAR = 2023
URL = f'https://indian-supreme-court-judgments.s3.amazonaws.com/data/tar/year={YEAR}/english/english.tar'
!curl -sI "{URL}" | grep -iE "HTTP/|content-length"
!wget -q --show-progress -O /content/english.tar "{URL}"
!mkdir -p /content/raw_all && tar -xf /content/english.tar -C /content/raw_all
import glob
pdfs = sorted(glob.glob('/content/raw_all/**/*.pdf', recursive=True))
print(len(pdfs), 'PDFs extracted')

## 5. Pick a reproducible sample
Keeps the corpus at a size you can label and index quickly. The list of chosen files is saved to Drive so you (or an interviewer) can rebuild the exact corpus.

In [ ]:
import random, shutil, os
N_DOCS = 700
SEED = 42
chosen = sorted(random.Random(SEED).sample(pdfs, min(N_DOCS, len(pdfs))))
shutil.rmtree('/content/raw', ignore_errors=True)
os.makedirs('/content/raw')
for p in chosen:
    shutil.copy(p, '/content/raw/')
os.makedirs('data', exist_ok=True)
with open('data/corpus_sample.txt', 'w') as f:
    f.write(f'# source: {URL}\n# seed: {SEED}, n: {len(chosen)}\n')
    f.writelines(os.path.basename(p) + '\n' for p in chosen)
print('copied', len(chosen), 'PDFs; list saved to data/corpus_sample.txt')

## 6. Ingest: clean the PDFs into `data/corpus`
A few *scanned PDF?* warnings are normal. If most files warn, tell Claude: that year is scanned images and we'll pick another.

In [ ]:
!python -m groundtruth ingest --src /content/raw --out data/corpus 2>&1 | tail -15
import statistics
from groundtruth.corpus import load_corpus
docs = load_corpus('data/corpus')
lens = [len(t) for t in docs.values()]
print(f'{len(docs)} documents | median {statistics.median(lens):,.0f} chars | total {sum(lens)/1e6:.1f}M chars')
first = next(iter(docs))
print('\nSample of', first, ':\n', docs[first][:600])

## 7. Day 2 — baseline retriever sanity check
Builds the dense baseline (`configs/baseline.yaml`) and shows the top 3 chunks for five hand-written questions. **Read them.** If results look random, stop and tell Claude before building the golden set. Embeddings are cached in Drive, so later runs are fast.

In [ ]:
from groundtruth.config import load_config
from groundtruth.pipeline import build_pipeline

pipe = build_pipeline(load_config('configs/baseline.yaml'))
print(len(pipe.chunks), 'chunks indexed')

questions = [
    'Is a demand notice required before prosecuting someone for a bounced cheque?',
    'When should bail be granted to an undertrial who has been in custody for years?',
    'Can the High Court quash criminal proceedings to prevent abuse of process?',
    'How is compensation calculated for a death in a motor accident?',
    'Does denying an employee the chance to cross-examine witnesses invalidate a disciplinary inquiry?',
]
for q in questions:
    _, top = pipe.search(q)
    print('\n=== Q:', q)
    for rank, (chunk, score) in enumerate(top[:3], 1):
        print(f'  {rank}. [{score:.3f}] {chunk.doc_id}: {chunk.text[:220]}...')

## Next: Days 3–4, the golden set
Run the cell below to sample 150 candidate passages, then come back to Claude: we'll write the questions together.

In [ ]:
!python -m groundtruth sample --n 150
!head -c 1200 data/candidates.jsonl